In [ ]:
#OCR with API

from dotenv import load_dotenv
from google import genai
import os
import base64
import time
import shutil
import zipfile
from pathlib import Path

# Load API key
load_dotenv()
client = genai.Client(api_key=os.getenv("GOOGLE_API_KEY"))

# --- Input: zip archive ---
zip_path = Path("Rassegna 1969.zip")
input_folder = Path("_temp_extracted_images")
output_folder = Path("ocr_output_Gemini_1969")
output_folder.mkdir(exist_ok=True)


# Extract zip
if input_folder.exists():
    shutil.rmtree(input_folder)
input_folder.mkdir()
print(f"Extracting {zip_path.name}...")
with zipfile.ZipFile(zip_path, "r") as zf:
    zf.extractall(input_folder)
print("Extraction complete.")

# Rate limiting config
BATCH_SIZE = 3
MAX_RPM = 20
MIN_INTERVAL = 60 / MAX_RPM  # 3 seconds between requests

supported = {".jpg", ".jpeg", ".png", ".webp", ".tif", ".tiff"}

# Collect images recursively (handles zips that unpack into a subfolder)
images = sorted([f for f in input_folder.rglob("*") if f.suffix.lower() in supported])
print(f"Found {len(images)} images to process.")

def get_mime_type(path: Path) -> str:
    suffix = path.suffix.lower()
    if suffix in {".jpg", ".jpeg"}:
        return "image/jpeg"
    elif suffix in {".tif", ".tiff"}:
        return "image/tiff"
    elif suffix == ".webp":
        return "image/webp"
    else:
        return "image/png"

def all_done(batch: list[Path]) -> bool:
    return all((output_folder / (p.stem + ".txt")).exists() for p in batch)

def process_batch(batch: list[Path], batch_num: int, total_batches: int):
    page_labels = ", ".join(p.name for p in batch)
    print(f"[Batch {batch_num}/{total_batches}] Processing: {page_labels}")

    parts = []
    for image_path in batch:
        image_data = base64.b64encode(image_path.read_bytes()).decode("utf-8")
        parts.append({"inline_data": {"mime_type": get_mime_type(image_path), "data": image_data}})

    n = len(batch)
    parts.append({
        "text": (
            f"The following {'image contains 1 page' if n == 1 else f'{n} images contain {n} pages'} from a book.\n"
            f"Transcribe all text from {'it' if n == 1 else 'each page'} exactly as it appears.\n"
            "Preserve the layout, column structure, and all numbers precisely.\n"
            "Reproduce the paragraph structure and do not break lines artificially.\n"
            + (
                "" if n == 1 else
                f"Separate each page's transcription with a delimiter on its own line: ===PAGE_BREAK===\n"
                f"Output exactly {n} sections separated by {n - 1} such delimiters."
            )
        )
    })

    response = client.models.generate_content(
        model="gemini-3.1-flash-lite",
        contents=[{"parts": parts}]
    )

    raw = response.text

    if n == 1:
        sections = [raw.strip()]
    else:
        sections = [s.strip() for s in raw.split("===PAGE_BREAK===")]

    if len(sections) != n:
        print(f"    ⚠ Expected {n} sections, got {len(sections)} — saving raw output to fallback file")
        fallback_path = output_folder / (batch[0].stem + "_batch_raw.txt")
        fallback_path.write_text(raw, encoding="utf-8")
        return

    for image_path, text in zip(batch, sections):
        out_path = output_folder / (image_path.stem + ".txt")
        out_path.write_text(text, encoding="utf-8")
        print(f"    → Saved {out_path.name}")

# Build batches
batches = [images[i:i + BATCH_SIZE] for i in range(0, len(images), BATCH_SIZE)]
total_batches = len(batches)
print(f"Total batches: {total_batches} (up to {BATCH_SIZE} pages each)")

last_request_time = 0.0

for batch_num, batch in enumerate(batches, 1):
    if all_done(batch):
        print(f"[Batch {batch_num}/{total_batches}] Skipping (all pages already done)")
        continue

    elapsed = time.time() - last_request_time
    wait = MIN_INTERVAL - elapsed
    if wait > 0:
        time.sleep(wait)

    try:
        last_request_time = time.time()
        process_batch(batch, batch_num, total_batches)
    except Exception as e:
        print(f"    ✗ Error on batch {batch_num}: {e}")

# Cleanup temp folder
shutil.rmtree(input_folder)
print("Temp folder cleaned up.")
print("Done!")

hello. Does this work?


7